# 05 — Distributed Residue Consistency

**Local constraints, noisy links, global structure**

This notebook extends `residue-manifold-learning` with a small bridge experiment inspired by distributed fault-tolerant quantum computing (FTQC), including IonQ's *Walking Cat* architecture.

The goal is not to simulate trapped-ion hardware directly.  
The goal is to build a minimal constraint-system analogue:

> local systems can remain valid while noisy links break global consistency.

## Bridge idea

- **Walking Cat / FTQC:** local logical modules are stabilized by code constraints, syndrome extraction, decoding, and resource factories.
- **Residue Manifold Learning (RML):** local residue systems are stabilized by modular constraints and scored through coverage / consistency metrics.
- **Distributed extension:** when modules are connected, link reliability becomes central.

```text
local modules → noisy links → global consistency
```

## Notebook outputs

This notebook writes:

```text
figures/distributed_residue_graph_clean.png
figures/distributed_residue_graph_noisy.png
figures/cgcs_noise_sweep.png
figures/global_consistency_heatmap.png

results/distributed_residue_consistency.csv
results/distributed_residue_summary.json
docs/notebook_05_distributed_residue_consistency.md
```

In [ ]:
# Setup

from pathlib import Path
import json
import math
import random
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import networkx as nx
except ImportError:
    !pip -q install networkx
    import networkx as nx

SEED = 9423
random.seed(SEED)
np.random.seed(SEED)

FIG_DIR = Path("figures")
RESULTS_DIR = Path("results")
DOCS_DIR = Path("docs")

for d in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Ready.")
print(f"seed = {SEED}")

## 1. Local residue manifold

For a mod30 prime-residue baseline, admissible residues are:

```text
{1, 7, 11, 13, 17, 19, 23, 29}
```

This does not mean every number in these lanes is prime.  
It means primes greater than 5 must persist inside these admissible residue lanes after excluding multiples of 2, 3, and 5.

In [ ]:
MODULUS = 30
ADMISSIBLE_RESIDUES = np.array([1, 7, 11, 13, 17, 19, 23, 29], dtype=int)
ADMISSIBLE_SET = set(ADMISSIBLE_RESIDUES.tolist())

def residue(x, modulus=MODULUS):
    return int(x % modulus)

def is_admissible_residue(r):
    return int(r % MODULUS) in ADMISSIBLE_SET

def sample_local_residues(n_samples=128, noise=0.0):
    """Sample a local residue system."""
    inadmissible = np.array([r for r in range(MODULUS) if r not in ADMISSIBLE_SET])
    values = []
    for _ in range(n_samples):
        if np.random.rand() < noise:
            values.append(int(np.random.choice(inadmissible)))
        else:
            values.append(int(np.random.choice(ADMISSIBLE_RESIDUES)))
    return np.array(values, dtype=int)

def local_coverage_score(samples):
    """Coverage = fraction of admissible residue lanes represented at least once."""
    present = set([int(x % MODULUS) for x in samples if is_admissible_residue(x)])
    return len(present.intersection(ADMISSIBLE_SET)) / len(ADMISSIBLE_RESIDUES)

def local_validity_score(samples):
    """Validity = fraction of samples that land in admissible residues."""
    return float(np.mean([is_admissible_residue(x) for x in samples]))

demo = sample_local_residues(n_samples=32, noise=0.0)
print("demo samples:", demo[:16])
print("coverage:", local_coverage_score(demo))
print("validity:", local_validity_score(demo))

## 2. Distributed graph

Each graph node is a local residue module.

Each edge is a link between modules.  
A link is consistent when paired module residues satisfy a simple compatibility rule.

For this first bridge notebook, the rule is intentionally simple:

```text
(r_i - r_j) mod 30 ∈ allowed differences
```

The allowed differences are generated from the admissible residues themselves.

In [ ]:
def allowed_difference_set(residues=ADMISSIBLE_RESIDUES, modulus=MODULUS):
    """Pairwise residue differences among admissible residues."""
    diffs = set()
    for a in residues:
        for b in residues:
            diffs.add(int((a - b) % modulus))
    return diffs

ALLOWED_DIFFS = allowed_difference_set()
print("allowed differences:", sorted(ALLOWED_DIFFS))
print("count:", len(ALLOWED_DIFFS))

def make_module_graph(n_modules=10, k_neighbors=3, rewiring=0.15, seed=SEED):
    """Create a small-world-ish module graph."""
    if n_modules < 4:
        raise ValueError("n_modules must be >= 4")
    k = min(k_neighbors, n_modules - 1)
    if k % 2 == 1:
        k += 1
    return nx.watts_strogatz_graph(n=n_modules, k=k, p=rewiring, seed=seed)

def assign_node_residue_samples(G, n_samples=128, local_noise=0.0):
    """Attach local residue samples and local scores to graph nodes."""
    for node in G.nodes:
        samples = sample_local_residues(n_samples=n_samples, noise=local_noise)
        G.nodes[node]["samples"] = samples
        G.nodes[node]["representative"] = int(np.random.choice(samples))
        G.nodes[node]["coverage"] = local_coverage_score(samples)
        G.nodes[node]["validity"] = local_validity_score(samples)
    return G

G_clean = make_module_graph(n_modules=12, k_neighbors=4, rewiring=0.15, seed=SEED)
G_clean = assign_node_residue_samples(G_clean, n_samples=128, local_noise=0.0)

print("nodes:", G_clean.number_of_nodes())
print("edges:", G_clean.number_of_edges())
print("mean local coverage:", np.mean([G_clean.nodes[n]["coverage"] for n in G_clean.nodes]))

## 3. Link noise and consistency

A noisy link may report a corrupted residue relationship between connected nodes.

This notebook separates:

- **local validity**: each node preserves admissible residue structure,
- **link consistency**: connected nodes maintain compatible relationships,
- **global stability**: consistent links form a large connected backbone.

In [ ]:
def evaluate_links(G, link_noise=0.0, allowed_diffs=ALLOWED_DIFFS, seed=None):
    """Mark each edge as consistent or inconsistent."""
    rng = np.random.default_rng(seed)

    for u, v in G.edges:
        ru = int(G.nodes[u]["representative"] % MODULUS)
        rv = int(G.nodes[v]["representative"] % MODULUS)

        corrupted = bool(rng.random() < link_noise)
        observed_rv = rv
        if corrupted:
            observed_rv = int(rng.integers(0, MODULUS))

        diff = int((ru - observed_rv) % MODULUS)
        consistent = diff in allowed_diffs

        G.edges[u, v]["corrupted"] = corrupted
        G.edges[u, v]["observed_diff"] = diff
        G.edges[u, v]["consistent"] = bool(consistent)

    return G

def link_consistency_score(G):
    if G.number_of_edges() == 0:
        return 1.0
    return float(np.mean([G.edges[e]["consistent"] for e in G.edges]))

def global_stability_score(G):
    """Global stability = size fraction of largest component formed by consistent links."""
    H = nx.Graph()
    H.add_nodes_from(G.nodes)
    H.add_edges_from([e for e in G.edges if G.edges[e]["consistent"]])

    if H.number_of_nodes() == 0:
        return 0.0

    largest = max((len(c) for c in nx.connected_components(H)), default=0)
    return float(largest / H.number_of_nodes())

def cgcs_score(G):
    local_coverage = float(np.mean([G.nodes[n]["coverage"] for n in G.nodes]))
    local_validity = float(np.mean([G.nodes[n]["validity"] for n in G.nodes]))
    link_consistency = link_consistency_score(G)
    global_stability = global_stability_score(G)

    cgcs = local_coverage * local_validity * link_consistency * global_stability

    return {
        "local_coverage": local_coverage,
        "local_validity": local_validity,
        "link_consistency": link_consistency,
        "global_stability": global_stability,
        "cgcs": float(cgcs),
    }

G_clean = evaluate_links(G_clean, link_noise=0.0, seed=SEED)
cgcs_score(G_clean)

## 4. Visualize clean vs noisy distributed systems

Blue edges are consistent.  
Dashed edges are inconsistent / corrupted.

The figure is intentionally visual-first for use in README, docs, and social posts.

In [ ]:
def draw_distributed_graph(G, title, out_path, seed=SEED):
    pos = nx.spring_layout(G, seed=seed, k=0.85)

    consistent_edges = [e for e in G.edges if G.edges[e].get("consistent", True)]
    inconsistent_edges = [e for e in G.edges if not G.edges[e].get("consistent", True)]

    node_scores = [G.nodes[n].get("coverage", 1.0) for n in G.nodes]

    plt.figure(figsize=(9, 6))
    nx.draw_networkx_nodes(
        G, pos,
        node_size=720,
        node_color=node_scores,
        cmap="Blues",
        vmin=0,
        vmax=1,
        linewidths=1.2,
        edgecolors="black",
    )
    nx.draw_networkx_edges(G, pos, edgelist=consistent_edges, width=2.0, alpha=0.8)
    nx.draw_networkx_edges(G, pos, edgelist=inconsistent_edges, width=2.0, alpha=0.8, style="dashed")

    labels = {n: f"M{n}" for n in G.nodes}
    nx.draw_networkx_labels(G, pos, labels=labels, font_size=9)

    scores = cgcs_score(G)
    subtitle = (
        f"local={scores['local_coverage']:.2f} | "
        f"links={scores['link_consistency']:.2f} | "
        f"global={scores['global_stability']:.2f} | "
        f"CGCS={scores['cgcs']:.2f}"
    )

    plt.title(title + "\n" + subtitle)
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.show()

G_clean = make_module_graph(n_modules=12, k_neighbors=4, rewiring=0.15, seed=SEED)
G_clean = assign_node_residue_samples(G_clean, n_samples=128, local_noise=0.0)
G_clean = evaluate_links(G_clean, link_noise=0.0, seed=SEED)

G_noisy = make_module_graph(n_modules=12, k_neighbors=4, rewiring=0.15, seed=SEED)
G_noisy = assign_node_residue_samples(G_noisy, n_samples=128, local_noise=0.0)
G_noisy = evaluate_links(G_noisy, link_noise=0.45, seed=SEED + 1)

draw_distributed_graph(
    G_clean,
    "Distributed residue graph: clean links",
    FIG_DIR / "distributed_residue_graph_clean.png",
)

draw_distributed_graph(
    G_noisy,
    "Distributed residue graph: noisy links",
    FIG_DIR / "distributed_residue_graph_noisy.png",
)

## 5. Noise sweep

Now sweep link noise and local noise.

This tests the central bridge claim:

> local constraint validity can remain high while distributed consistency drops through noisy links.

In [ ]:
def run_single_experiment(
    n_modules=12,
    k_neighbors=4,
    rewiring=0.15,
    n_samples=128,
    local_noise=0.0,
    link_noise=0.0,
    seed=SEED,
):
    G = make_module_graph(n_modules=n_modules, k_neighbors=k_neighbors, rewiring=rewiring, seed=seed)
    G = assign_node_residue_samples(G, n_samples=n_samples, local_noise=local_noise)
    G = evaluate_links(G, link_noise=link_noise, seed=seed + 99)
    return cgcs_score(G)

def run_noise_sweep(
    link_noise_values=np.linspace(0, 0.7, 15),
    local_noise_values=(0.0, 0.05, 0.10),
    repeats=25,
    n_modules=12,
):
    rows = []
    for local_noise in local_noise_values:
        for link_noise in link_noise_values:
            for rep in range(repeats):
                scores = run_single_experiment(
                    n_modules=n_modules,
                    local_noise=float(local_noise),
                    link_noise=float(link_noise),
                    seed=SEED + rep * 1000 + int(link_noise * 1000) + int(local_noise * 10000),
                )
                rows.append({
                    "local_noise": float(local_noise),
                    "link_noise": float(link_noise),
                    "repeat": rep,
                    **scores,
                })
    return pd.DataFrame(rows)

df = run_noise_sweep()
csv_path = RESULTS_DIR / "distributed_residue_consistency.csv"
df.to_csv(csv_path, index=False)

print(df.head())
print(f"saved: {csv_path}")

In [ ]:
summary = (
    df.groupby(["local_noise", "link_noise"], as_index=False)
    .agg({
        "local_coverage": "mean",
        "local_validity": "mean",
        "link_consistency": "mean",
        "global_stability": "mean",
        "cgcs": "mean",
    })
)

summary.head()

In [ ]:
def plot_noise_sweep(summary_df, local_noise=0.0):
    sub = summary_df[summary_df["local_noise"] == local_noise].copy()

    plt.figure(figsize=(9, 5.5))
    plt.plot(sub["link_noise"], sub["local_coverage"], marker="o", label="local coverage")
    plt.plot(sub["link_noise"], sub["local_validity"], marker="o", label="local validity")
    plt.plot(sub["link_noise"], sub["link_consistency"], marker="o", label="link consistency")
    plt.plot(sub["link_noise"], sub["global_stability"], marker="o", label="global stability")
    plt.plot(sub["link_noise"], sub["cgcs"], marker="o", linewidth=3, label="CGCS")

    plt.xlabel("link noise")
    plt.ylabel("score")
    plt.ylim(-0.02, 1.02)
    plt.title(f"Distributed residue consistency under link noise (local_noise={local_noise})")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    out = FIG_DIR / "cgcs_noise_sweep.png"
    plt.savefig(out, dpi=180, bbox_inches="tight")
    plt.show()
    return out

plot_noise_sweep(summary, local_noise=0.0)

## 6. Heatmap: link noise × local noise

This heatmap shows how CGCS changes as local residue corruption and link corruption vary together.

In [ ]:
pivot = summary.pivot(index="local_noise", columns="link_noise", values="cgcs")

plt.figure(figsize=(9, 4.8))
plt.imshow(pivot.values, aspect="auto", origin="lower")
plt.colorbar(label="CGCS")
plt.xticks(range(len(pivot.columns)), [f"{x:.2f}" for x in pivot.columns], rotation=45)
plt.yticks(range(len(pivot.index)), [f"{x:.2f}" for x in pivot.index])
plt.xlabel("link noise")
plt.ylabel("local noise")
plt.title("Global consistency heatmap")
plt.tight_layout()
heatmap_path = FIG_DIR / "global_consistency_heatmap.png"
plt.savefig(heatmap_path, dpi=180, bbox_inches="tight")
plt.show()

print(f"saved: {heatmap_path}")

## 7. Summary JSON

Export a compact summary for README/docs automation.

In [ ]:
baseline = summary[(summary["local_noise"] == 0.0) & (summary["link_noise"] == 0.0)].iloc[0].to_dict()
high_link_noise = summary[(summary["local_noise"] == 0.0) & (summary["link_noise"] == summary["link_noise"].max())].iloc[0].to_dict()

summary_payload = {
    "notebook": "05_distributed_residue_consistency.ipynb",
    "seed": SEED,
    "modulus": MODULUS,
    "admissible_residues": ADMISSIBLE_RESIDUES.tolist(),
    "allowed_differences": sorted(list(ALLOWED_DIFFS)),
    "core_claim": "Local residue validity can remain high while global consistency drops through noisy links.",
    "cgcs_definition": "CGCS = local_coverage × local_validity × link_consistency × global_stability",
    "baseline": {k: float(v) if isinstance(v, (int, float, np.floating)) else v for k, v in baseline.items()},
    "high_link_noise": {k: float(v) if isinstance(v, (int, float, np.floating)) else v for k, v in high_link_noise.items()},
    "figures": [
        "figures/distributed_residue_graph_clean.png",
        "figures/distributed_residue_graph_noisy.png",
        "figures/cgcs_noise_sweep.png",
        "figures/global_consistency_heatmap.png",
    ],
    "results": [
        "results/distributed_residue_consistency.csv",
    ],
}

summary_path = RESULTS_DIR / "distributed_residue_summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")

print(json.dumps(summary_payload, indent=2)[:1200] + "...")
print(f"saved: {summary_path}")

## 8. Markdown export

Create a short markdown summary that can be copied into `docs/` or linked from the README.

In [ ]:
md_text = """# Notebook 05 — Distributed Residue Consistency

**Core claim:** local residue validity can remain high while global consistency drops through noisy links.

## Bridge

Walking Cat-style distributed FTQC suggests a useful constraint-system question:

> when local modules are stable, how much does global coherence depend on links?

RML analogue:

```text
mod30 local residues
→ residue consistency across links
→ graph-level structure persistence
```

## CGCS bridge definition

```text
CGCS = local_coverage × local_validity × link_consistency × global_stability
```

## Outputs

- `figures/distributed_residue_graph_clean.png`
- `figures/distributed_residue_graph_noisy.png`
- `figures/cgcs_noise_sweep.png`
- `figures/global_consistency_heatmap.png`
- `results/distributed_residue_consistency.csv`
- `results/distributed_residue_summary.json`

## Takeaway

Local constraint validity is necessary but not sufficient for distributed consistency.

In distributed systems, links become part of the constraint manifold.
"""

md_path = DOCS_DIR / "notebook_05_distributed_residue_consistency.md"
md_path.write_text(md_text, encoding="utf-8")

print(md_text)
print(f"saved: {md_path}")

## 9. Optional zip/export block for Colab

Uncomment the final line when running in Google Colab.

In [ ]:
# Optional: create a zip of generated outputs.

import zipfile

zip_path = Path("notebook_05_outputs.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for folder in [FIG_DIR, RESULTS_DIR, DOCS_DIR]:
        for path in folder.rglob("*"):
            if path.is_file():
                z.write(path, path.as_posix())

print(f"created: {zip_path}")

# Optional Colab download:
# from google.colab import files
# files.download(str(zip_path))

## Final interpretation

This notebook gives `residue-manifold-learning` a small bridge to distributed FTQC:

```text
single module: local validity
many modules: link consistency
full system: global constraint stability
```

That is the minimal structure needed for the Walking Cat bridge:

```text
Walking Cat = dynamic constraint manifold
RML = static residue constraint manifold
distributed scaling = consistency across links
```